In [7]:
import random

def get_fitness(times, buses, w1, w2, max_buses):
    wait = sum((t - (i * 10))**2 for i, t in enumerate(times))
    cost = len(set(buses)) * 500
    penalty = 0
    for i in range(len(times)):
        for j in range(i + 1, len(times)):
            if buses[i] == buses[j]:
                if abs(times[i] - times[j]) < 30.0:
                    penalty += 10000
    return (w1 * wait) + (w2 * cost) + penalty

num_trips = int(input("Trips: "))
max_buses = int(input("Max Buses: "))
N = int(input("Particles: "))
T = int(input("Iterations: "))
w1 = float(input("w1: "))
w2 = float(input("w2: "))

w, c1, c2 = 0.7, 1.5, 1.5

pos_t = [sorted([random.uniform(0, 120) for _ in range(num_trips)]) for _ in range(N)]
pos_b = [[random.randint(1, max_buses) for _ in range(num_trips)] for _ in range(N)]
vel_t = [[random.uniform(-5, 5) for _ in range(num_trips)] for _ in range(N)]
vel_b = [[random.uniform(-1, 1) for _ in range(num_trips)] for _ in range(N)]

p_best_t = [list(t) for t in pos_t]
p_best_b = [list(b) for b in pos_b]
p_score = [get_fitness(pos_t[i], pos_b[i], w1, w2, max_buses) for i in range(N)]

g_idx = p_score.index(min(p_score))
g_best_t, g_best_b, g_score = list(pos_t[g_idx]), list(pos_b[g_idx]), p_score[g_idx]

for _ in range(T):
    for i in range(N):
        r1, r2 = random.random(), random.random()
        for k in range(num_trips):
            vel_t[i][k] = w * vel_t[i][k] + c1 * r1 * (p_best_t[i][k] - pos_t[i][k]) + c2 * r2 * (g_best_t[k] - pos_t[i][k])
            pos_t[i][k] += vel_t[i][k]

            vel_b[i][k] = w * vel_b[i][k] + c1 * r1 * (p_best_b[i][k] - pos_b[i][k]) + c2 * r2 * (g_best_b[k] - pos_b[i][k])
            pos_b[i][k] += vel_b[i][k]

        pos_t[i] = sorted(pos_t[i])
        pos_b[i] = [round(max(1, min(b, max_buses))) for b in pos_b[i]]

        score = get_fitness(pos_t[i], pos_b[i], w1, w2, max_buses)
        if score < p_score[i]:
            p_best_t[i], p_best_b[i], p_score[i] = list(pos_t[i]), list(pos_b[i]), score
            if score < g_score:
                g_best_t, g_best_b, g_score = list(pos_t[i]), list(pos_b[i]), score

print("\n--- RESULTS ---")
print(f"Best Score: {g_score:.2f}")
print("Times:", [round(t, 1) for t in g_best_t])
print("Buses:", g_best_b)


Trips: 10
Max Buses: 6
Particles: 50
Iterations: 150
w1: 0.3
w2: 0.7

--- RESULTS ---
Best Score: 1402.78
Times: [-1.3, 8.9, 20.0, 30.1, 38.9, 50.0, 59.5, 69.5, 79.6, 92.1]
Buses: [4, 2, 3, 4, 2, 3, 6, 2, 4, 3]
